<a href="https://colab.research.google.com/github/ldaniel-hm/eml_tabular/blob/main/MonteCarlo_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Monte Carlo On-Policy y Off-Policy en Taxi-v3**

_Estudio comparativo de control Monte Carlo tabular con muestreo por importancia ponderado_

Este notebook implementa y compara dos algoritmos de control Monte Carlo sobre el entorno **Taxi-v3** de Gymnasium:

- **On-Policy (Algoritmo 3, todas las visitas):** la política de exploración coincide con la que se mejora (ε-greedy). La actualización de Q se realiza con el retorno del episodio actual, sin corrección de importancia.
- **Off-Policy (Algoritmo 6, IS ponderado):** dos políticas distintas — política de comportamiento $b$ (ε-greedy) para generar datos y política objetivo $\pi$ (greedy pura) que se mejora. El ratio de importancia $W = \pi(A_t|S_t)/b(A_t|S_t)$ escala cada retorno; el bucle de retropropagación se interrumpe en cuanto se toma una acción no-greedy (porque $\pi(a|s)=0$ y $W=0$).

### ¿Por qué Taxi-v3?

FrozenLake ofrece recompensas binarias (0 ó 1). Taxi-v3 introduce una estructura más rica:

- **500 estados discretos**: 25 posiciones del taxi × 5 ubicaciones del pasajero × 4 destinos.
- **6 acciones**: South(0), North(1), East(2), West(3), Pickup(4), Dropoff(5).
- **Recompensas**: −1 por paso, +20 entrega exitosa, −10 acción ilegal (pickup/dropoff en posición incorrecta).
- **Señal de recompensa escasa y ruidosa**: las acciones Pickup/Dropoff son determinantes pero tienen alta penalización si se ejecutan incorrectamente.

Este entorno permite estudiar cómo on-policy y off-policy difieren cuando hay acciones con consecuencias asimétricas y la política de exploración genera experiencias alejadas de la política objetivo.

## **1. Preparación del Entorno**

La preparación consta de las siguientes partes:
- **Instalación de dependencias**: librería `gymnasium`.
- **Importación de librerías**: `numpy`, `matplotlib`, `tqdm`.
- **Creación del entorno Taxi-v3**: espacio de observación `Discrete(500)`, espacio de acciones `Discrete(6)`, `render_mode='ansi'` para visualización en texto.

##### _________ **Código de la Instalación e Importación**
----

In [ ]:
%%capture
#@title Instalamos gymnasium
!pip install gymnasium

In [ ]:
#@title Importamos librerias
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import gymnasium as gym

In [ ]:
#@title Importamos el entorno Taxi-v3
env = gym.make('Taxi-v3', render_mode='ansi')

nS = env.observation_space.n  # 500
nA = env.action_space.n       # 6
print(f"Estados: {nS}, Acciones: {nA}")
print(f"Recompensas: -1 por paso | +20 entrega exitosa | -10 acción ilegal")
print()
# Estado inicial de ejemplo
obs, info = env.reset(seed=42)
print(f"Estado inicial (seed=42): {obs}")
taxi_row, taxi_col, pass_loc, dest_idx = env.unwrapped.decode(obs)
locs = ['R', 'G', 'Y', 'B', 'en taxi']
print(f"  Taxi: ({taxi_row},{taxi_col}), Pasajero: {locs[pass_loc]}, Destino: {locs[dest_idx]}")
print(env.render())

## **2. Diseño del Agente**

El diseño del agente sigue el esquema de Gymnasium (sección 5.2): `__init__`, `get_action`, `update`, `stats`.

### Políticas
- **ε-soft** (`random_epsilon_greedy_policy`): distribución base con $\pi(a|s) \geq \varepsilon/|A|$ para todas las acciones.
- **ε-greedy** (`epsilon_greedy_policy`): muestrea desde la ε-soft. Usada por la política de comportamiento $b$.

### Agentes
- **`MonteCarloOnPolicyAgent`**: actualiza Q con retornos calculados hacia atrás al final del episodio (todas las visitas). Media incremental: $Q(s,a) \leftarrow Q(s,a) + \frac{1}{n(s,a)}[G - Q(s,a)]$.
- **`MonteCarloOffPolicyAgent`**: aplica el Algoritmo 6 de Sutton & Barto. Bucle inverso que acumula $C(s,a) \mathrel{+}= W$ y actualiza $Q(s,a) \mathrel{+}= \frac{W}{C(s,a)}[G - Q(s,a)]$; se interrumpe cuando $A_t \neq \arg\max Q[s]$.

Ambos agentes comparten la misma función `train()` con el bucle episódico estándar.

#### **Código de las políticas y algoritmos MC**
----------------

In [ ]:
# @title Políticas del agente

# Acciones del entorno Taxi-v3
SOUTH, NORTH, EAST, WEST, PICKUP, DROPOFF = 0, 1, 2, 3, 4, 5

def random_epsilon_greedy_policy(Q, epsilon, state, nA):
    """Distribución epsilon-soft: probabilidad mínima epsilon/nA para cada acción."""
    pi_A = np.ones(nA, dtype=float) * epsilon / nA
    best_action = np.argmax(Q[state])
    pi_A[best_action] += (1.0 - epsilon)
    return pi_A

def epsilon_greedy_policy(Q, epsilon, state, nA):
    """Muestrea una acción de la distribución epsilon-soft."""
    pi_A = random_epsilon_greedy_policy(Q, epsilon, state, nA)
    return np.random.choice(np.arange(nA), p=pi_A)

In [ ]:
#@title Agente Monte Carlo On-Policy (esquema Gymnasium, sección 5.2)

class MonteCarloOnPolicyAgent:
    """
    Agente Monte Carlo On-Policy con criterio de todas las visitas.
    Sigue el esquema de Gymnasium: __init__, get_action, update, stats.

    La política de exploración y la política objetivo son la misma (epsilon-greedy).
    Actualiza Q con media incremental: Q(s,a) += (1/n(s,a)) * (G - Q(s,a)).
    Los retornos se calculan hacia atrás al final de cada episodio.
    """

    def __init__(self, env, epsilon=0.2, decay=False, discount_factor=1.0):
        """Inicializa todo lo necesario para el aprendizaje."""
        self.env = env
        self.epsilon = epsilon
        self.decay = decay
        self.discount_factor = discount_factor

        nA = env.action_space.n
        nS = env.observation_space.n
        self.Q = np.zeros([nS, nA])
        self.n_visits = np.zeros([nS, nA])

        self._episode = []       # (state, action, reward)
        self._episode_count = 0
        self._total_reward = 0.0
        self._list_stats = []    # recompensa media acumulada por episodio
        self._list_lengths = []  # longitud de cada episodio

    def get_action(self, state):
        """Acción según la política epsilon-greedy (política b = política objetivo)."""
        nA = self.env.action_space.n
        return epsilon_greedy_policy(self.Q, self.epsilon, state, nA)

    def update(self, obs, action, next_obs, reward, terminated, truncated, info):
        """
        Acumula (s, a, r) durante el episodio.
        Al terminar: calcula retornos G hacia atrás y actualiza Q (todas las visitas).
        """
        self._episode.append((obs, action, reward))

        if terminated or truncated:
            G = 0.0
            for (s, a, r) in reversed(self._episode):
                G = r + self.discount_factor * G
                self.n_visits[s, a] += 1.0
                alpha = 1.0 / self.n_visits[s, a]
                self.Q[s, a] += alpha * (G - self.Q[s, a])

            episode_reward = sum(r for _, _, r in self._episode)
            self._total_reward += episode_reward
            self._episode_count += 1
            self._list_lengths.append(len(self._episode))
            self._list_stats.append(self._total_reward / self._episode_count)

            if self.decay:
                self.epsilon = min(1.0, 1000.0 / (self._episode_count + 1))

            self._episode = []

    def stats(self):
        """Retorna recompensa media acumulada por episodio y longitudes."""
        return self._list_stats, self._list_lengths


def train(env, agent, num_episodes):
    """Bucle de entrenamiento episódico (esquema Gymnasium, sección 5.2)."""
    step_display = max(1, num_episodes // 10)
    for t in tqdm(range(num_episodes)):
        obs, info = env.reset()   # sin seed fijo: estados iniciales variados
        done = False
        while not done:
            action = agent.get_action(obs)
            next_obs, reward, terminated, truncated, info = env.step(action)
            agent.update(obs, action, next_obs, reward, terminated, truncated, info)
            done = terminated or truncated
            obs = next_obs
        if (t + 1) % step_display == 0:
            list_stats, _ = agent.stats()
            print(f"reward_medio: {list_stats[-1]:.2f}, epsilon: {agent.epsilon:.4f}")
    return agent.stats()

In [ ]:
#@title Agente Monte Carlo Off-Policy (esquema Gymnasium, sección 5.2)

class MonteCarloOffPolicyAgent:
    """
    Agente Monte Carlo Off-Policy con muestreo por importancia ponderado.
    Implementa el Algoritmo 6 de Sutton & Barto (2018).

    - Política de comportamiento b: epsilon-greedy sobre Q (genera los episodios).
    - Política objetivo pi: greedy sobre Q (la que se mejora).

    Bucle de retropropagación al finalizar cada episodio:
      C(s,a) += W
      Q(s,a) += (W/C(s,a)) * (G - Q(s,a))
      Si A_t != argmax Q[s]: break  (pi es determinista, W se haría 0)
      W *= 1/b(a|s)

    C(s,a) acumula los pesos de importancia (denominador del IS ponderado).
    """

    def __init__(self, env, epsilon=0.3, decay=False, discount_factor=1.0, q_init=None):
        """Inicializa todo lo necesario para el aprendizaje."""
        self.env = env
        self.epsilon = epsilon
        self.decay = decay
        self.discount_factor = discount_factor

        nA = env.action_space.n
        nS = env.observation_space.n

        if q_init is None:
            self.Q = np.zeros([nS, nA])
        else:
            rng = np.random.default_rng(42)
            self.Q = rng.uniform(0, q_init, [nS, nA])

        self.C = np.zeros([nS, nA])   # acumulador de pesos IS ponderado
        self._episode = []            # (state, action, reward, b_prob)
        self._episode_count = 0
        self._total_reward = 0.0
        self._list_stats = []
        self._list_lengths = []

    def get_action(self, state):
        """Acción según la política de comportamiento b (epsilon-greedy)."""
        nA = self.env.action_space.n
        return epsilon_greedy_policy(self.Q, self.epsilon, state, nA)

    def update(self, obs, action, next_obs, reward, terminated, truncated, info):
        """
        Acumula (s, a, r, b_prob). Al terminar el episodio aplica Algoritmo 6:
          - Retornos G hacia atrás
          - Actualización ponderada con C y W
          - Break cuando acción != argmax Q[s] (pi greedy determina W=0)
        """
        nA = self.env.action_space.n
        greedy_a = np.argmax(self.Q[obs])
        if action == greedy_a:
            b_prob = 1.0 - self.epsilon + self.epsilon / nA
        else:
            b_prob = self.epsilon / nA
        self._episode.append((obs, action, reward, b_prob))

        if terminated or truncated:
            G = 0.0
            W = 1.0
            for (s, a, r, bp) in reversed(self._episode):
                G = r + self.discount_factor * G
                self.C[s, a] += W
                self.Q[s, a] += (W / self.C[s, a]) * (G - self.Q[s, a])
                if a != np.argmax(self.Q[s]):
                    break
                W *= 1.0 / bp

            episode_reward = sum(r for _, _, r, _ in self._episode)
            self._total_reward += episode_reward
            self._episode_count += 1
            self._list_lengths.append(len(self._episode))
            self._list_stats.append(self._total_reward / self._episode_count)

            if self.decay:
                self.epsilon = min(1.0, 1000.0 / (self._episode_count + 1))

            self._episode = []

    def stats(self):
        """Retorna recompensa media acumulada por episodio y longitudes."""
        return self._list_stats, self._list_lengths

## **3. Experimentación**

### **3.1 Representaciones Gráficas**

**Gráfica 1 — Recompensa media por episodio:** $f(t) = \frac{\sum_{i=1}^{t} R_i}{t}$

En Taxi-v3, la recompensa por episodio va de muy negativa (agente aleatorio con acciones ilegales frecuentes, $\approx -200$) hasta positiva ($\approx +6$ para el recorrido óptimo, que tarda ~14 pasos). Esta curva mide directamente la calidad de la política aprendida.

---

**Gráfica 2 — Longitud de los episodios:** $f(t) = \text{len}(\text{episodio}_t)$

Se pueden observar las mismas 3 fases que en FrozenLake, pero con dinámicas específicas de Taxi:

1. **Inicio:** el agente actúa aleatoriamente. Frecuentes penalizaciones de −10 por Pickup/Dropoff ilegales; los episodios se truncan a 200 pasos sin entregar al pasajero. La media móvil parte de 200 pasos.

2. **Aprendizaje:** el agente aprende a evitar penalizaciones y a orientarse hacia el pasajero y el destino. Los episodios empiezan a terminar con entrega exitosa, reduciendo la longitud. La media móvil desciende.

3. **Convergencia:** los episodios exitosos se estabilizan en la longitud óptima (~14–16 pasos para la configuración promedio de Taxi). La media móvil se aplana cerca del óptimo.

---

**Función `show_greedy_episode`:** ejecuta un episodio con la política greedy ($\arg\max_a Q(s,a)$) desde un estado fijo (semilla 42) y muestra el estado inicial y final en formato ANSI. Permite verificar visualmente si el taxi ha aprendido a recoger y entregar al pasajero.

In [ ]:
# @title Funciones para mostrar los resultados

def plot(list_stats, title='Recompensa media por episodio', ylabel='Recompensa media'):
    """Evolución de la recompensa media acumulada por episodio."""
    indices = list(range(len(list_stats)))
    plt.figure(figsize=(8, 3))
    plt.plot(indices, list_stats)
    plt.title(title)
    plt.xlabel('Episodio')
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.show()

def plot_lengths(list_lengths, window=500):
    """Longitud de los episodios con curva de tendencia (media móvil)."""
    n = len(list_lengths)
    window = min(window, max(1, n // 10))
    plt.figure(figsize=(10, 4))
    plt.plot(list_lengths, alpha=0.25, color='steelblue',
             linewidth=0.8, label='Longitud del episodio')
    if n >= window and window > 1:
        moving_avg = np.convolve(list_lengths, np.ones(window) / window, mode='valid')
        plt.plot(range(window - 1, n), moving_avg,
                 color='crimson', linewidth=2,
                 label=f'Tendencia (media móvil, ventana={window})')
    plt.title('Longitud de los episodios')
    plt.xlabel('Episodio')
    plt.ylabel('Número de pasos')
    plt.legend()
    plt.grid(True)
    plt.show()

def show_greedy_episode(env, Q, max_steps=200, seed=42, title="Episodio greedy"):
    """
    Ejecuta un episodio con la política greedy (argmax Q) y muestra el estado
    inicial y el estado final en formato ANSI. Útil para verificar visualmente
    que el taxi recoge y entrega al pasajero correctamente.
    """
    state, info = env.reset(seed=seed)
    initial_frame = env.render()
    done = False
    total_reward = 0
    steps = 0
    while not done and steps < max_steps:
        action = np.argmax(Q[state])
        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated
    final_frame = env.render()
    if done and total_reward > -steps:   # entregó sin penalizaciones graves
        status = 'Éxito'
    elif not done:
        status = 'Truncado (no convergió)'
    else:
        status = 'Terminado'
    print(f"{title}  |  Recompensa: {total_reward:.0f}  |  Pasos: {steps}  |  {status}")
    print("\nEstado inicial:")
    print(initial_frame)
    print("Estado final:")
    print(final_frame)

def print_q_summary(env, Q, title="Resumen tabla Q"):
    """
    Resumen estadístico de la tabla Q aprendida.
    Para Taxi (500 x 6 = 3000 entradas) no es práctico imprimir la tabla completa;
    se muestran estadísticas globales y algunos estados decodificados.
    """
    non_zero = np.count_nonzero(Q)
    total = Q.size
    locs = ['R', 'G', 'Y', 'B', 'en taxi']
    actions_names = ['S', 'N', 'E', 'W', 'PU', 'DO']
    print(f"--- {title} ---")
    print(f"  Entradas no nulas : {non_zero}/{total} ({100*non_zero/total:.1f}%)")
    print(f"  Max Q             : {Q.max():.3f}")
    if non_zero > 0:
        print(f"  Min Q (no nulo)   : {Q[Q != 0].min():.3f}")
        print(f"  Q media (no nulo) : {Q[Q != 0].mean():.3f}")
    print("\n  Muestra de estados:")
    for state in [0, 100, 200, 300, 499]:
        r, c, pl, di = env.unwrapped.decode(state)
        q = Q[state]
        best = np.argmax(q)
        print(f"  [{state:3d}] taxi=({r},{c}) pas={locs[pl]:7s} dst={locs[di]:2s} "
              f"-> mejor={actions_names[best]} Q={q.round(2)}")

### **3.2 On-Policy en Taxi-v3**

Se entrenan **100 000 episodios** con $\varepsilon$ con decaimiento: $\varepsilon = \min(1.0,\ 1000/(t+1))$.

El decaimiento es necesario porque:
- Al inicio, $\varepsilon$ alto favorece exploración amplia (el taxi aprende a evitar penalizaciones).
- Al final, $\varepsilon \to 0$ hace que la política de comportamiento se acerque a la greedy y los episodios sean más cortos y eficientes.

La semilla numpy fija (`np.random.seed(42)`) garantiza reproducibilidad.

In [ ]:
# @title Aprendizaje on-policy
np.random.seed(42)
agent_on = MonteCarloOnPolicyAgent(env, epsilon=0.2, decay=True, discount_factor=1.0)
list_stats_on, list_lengths_on = train(env, agent_on, num_episodes=100000)
Q_on = agent_on.Q

In [ ]:
#@title Recompensa media por episodio (on-policy)
plot(list_stats_on, title='On-Policy — Recompensa media por episodio')
print(f'Recompensa media final: {list_stats_on[-1]:.2f}')

In [ ]:
#@title Longitud de episodios (on-policy)
plot_lengths(list_lengths_on)
print(f'Longitud media final (últimos 1000 episodios): {np.mean(list_lengths_on[-1000:]):.2f} pasos')

####.
Resumen estadístico de la tabla Q aprendida. Con 500 estados y 6 acciones la tabla tiene 3000 entradas.
Se muestran estadísticas globales y algunos estados decodificados para verificar que el agente ha aprendido acciones coherentes.

In [ ]:
# @title Resumen tabla Q — on-policy
print_q_summary(env, Q_on, title="On-Policy — Tabla Q")

####.
Se ejecuta un episodio con la política greedy aprendida (estado fijo, semilla 42) para verificar visualmente
que el taxi recoge al pasajero en su ubicación y lo entrega en el destino correcto.

In [ ]:
#@title Episodio greedy (on-policy)
show_greedy_episode(env, Q_on, seed=42, title="On-Policy — Episodio greedy")

### **3.3 Off-Policy en Taxi-v3**

Se entrenan **100 000 episodios** con $\varepsilon$ con decaimiento.

Diferencias respecto al caso on-policy:

- **Bucle de retropropagación interrumpido**: solo la **cola greedy** final de cada episodio actualiza Q. En las primeras etapas, con $\varepsilon$ alto, casi todos los pasos son no-greedy (especialmente PICKUP y DROPOFF, que son raramente la acción greedy al inicio). Esto ralentiza el aprendizaje inicial.
- **Sin `q_init`**: Taxi-v3 inicializa los episodios desde estados aleatorios (300 configuraciones válidas), por lo que el sesgo de `argmax` hacia la primera acción no es sistemático como en FrozenLake 8×8.
- **Acciones ilegales y off-policy**: la penalización de −10 por PICKUP/DROPOFF ilegales genera retornos muy negativos para ciertos pares (s,a). Con IS ponderado, estos retornos solo actualizan Q cuando la acción ilegal coincide con la greedy, reduciendo la contaminación de la tabla Q.

In [ ]:
# @title Aprendizaje off-policy
np.random.seed(42)
agent_off = MonteCarloOffPolicyAgent(env, epsilon=0.3, decay=True, discount_factor=1.0)
list_stats_off, list_lengths_off = train(env, agent_off, num_episodes=100000)
Q_off = agent_off.Q

In [ ]:
#@title Recompensa media por episodio (off-policy)
plot(list_stats_off, title='Off-Policy — Recompensa media por episodio')
print(f'Recompensa media final: {list_stats_off[-1]:.2f}')

In [ ]:
#@title Longitud de episodios (off-policy)
plot_lengths(list_lengths_off)
print(f'Longitud media final (últimos 1000 episodios): {np.mean(list_lengths_off[-1000:]):.2f} pasos')

####.
Resumen estadístico de la tabla Q aprendida por el agente off-policy.
Se espera una cobertura menor de entradas no nulas respecto al on-policy, ya que el bucle de
retropropagación solo actualiza la cola greedy de cada episodio.

In [ ]:
# @title Resumen tabla Q — off-policy
print_q_summary(env, Q_off, title="Off-Policy — Tabla Q")

In [ ]:
#@title Episodio greedy (off-policy)
show_greedy_episode(env, Q_off, seed=42, title="Off-Policy — Episodio greedy")

### **3.4 Comparativa On-Policy vs Off-Policy**

Se representan en los mismos ejes las curvas de recompensa media y de longitud de episodios (media móvil) de ambos agentes. El objetivo es responder a las siguientes hipótesis:

1. **¿Cuál converge antes a recompensa positiva?** On-policy actualiza todos los estados visitados en cada episodio; off-policy solo la cola greedy. Se espera que on-policy sea más rápido en las primeras etapas.

2. **¿La política final es equivalente?** Si ambos convergen a $Q^*$, las curvas deberían cruzarse y estabilizarse en el mismo nivel. Una brecha persistente indicaría que el IS ponderado introduce sesgo o varianza adicional en Taxi.

3. **¿Difieren en la longitud del episodio convergido?** Una longitud menor en el off-policy indicaría que la política objetivo greedy es más eficiente (al no estar contaminada por la exploración ε-greedy).

In [ ]:
# @title Comparativa on-policy vs off-policy
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Recompensa media ---
axes[0].plot(list_stats_on,  alpha=0.85, color='steelblue', linewidth=1, label='On-policy')
axes[0].plot(list_stats_off, alpha=0.85, color='crimson',   linewidth=1, label='Off-policy')
axes[0].set_title('Recompensa media por episodio')
axes[0].set_xlabel('Episodio')
axes[0].set_ylabel('Recompensa media')
axes[0].legend()
axes[0].grid(True)

# --- Longitud (media móvil) ---
w = max(1, len(list_lengths_on) // 20)
ma_on  = np.convolve(list_lengths_on,  np.ones(w) / w, mode='valid')
ma_off = np.convolve(list_lengths_off, np.ones(w) / w, mode='valid')
axes[1].plot(range(w - 1, len(list_lengths_on)),  ma_on,  color='steelblue', linewidth=1.5, label='On-policy')
axes[1].plot(range(w - 1, len(list_lengths_off)), ma_off, color='crimson',   linewidth=1.5, label='Off-policy')
axes[1].set_title(f'Longitud media de episodios (tendencia, ventana={w})')
axes[1].set_xlabel('Episodio')
axes[1].set_ylabel('Pasos')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Estadísticas finales
print(f"On-policy  — Recompensa media final: {list_stats_on[-1]:.2f}  | "
      f"Longitud media final: {np.mean(list_lengths_on[-1000:]):.1f} pasos  | "
      f"epsilon final: {agent_on.epsilon:.4f}")
print(f"Off-policy — Recompensa media final: {list_stats_off[-1]:.2f}  | "
      f"Longitud media final: {np.mean(list_lengths_off[-1000:]):.1f} pasos  | "
      f"epsilon final: {agent_off.epsilon:.4f}")

## **4. Análisis y Estudios Futuros**

### **4.1 Análisis de Resultados**

- **On-policy** actualiza Q con todos los pares (s,a) visitados en cada episodio, con lo que converge antes en las etapas iniciales cuando la política es aleatoria y los episodios son largos.

- **Off-policy** solo actualiza la cola greedy del episodio (pasos desde el final hasta el primer paso no-greedy). Con $\varepsilon$ alto, esto puede ser solo 1–2 pasos por episodio. A medida que $\varepsilon$ decae, el backward pass alcanza más pasos y la eficiencia de datos mejora.

- La penalización de −10 por acciones ilegales afecta de forma distinta a cada agente: on-policy aprende rápido a evitarlas porque actualiza Q directamente desde los episodios donde ocurren; off-policy las ignora si la acción ilegal no era greedy en ese instante.

- El acumulador $C(s,a)$ del IS ponderado actúa como denominador de una media ponderada: da más peso a los retornos obtenidos cuando la política de comportamiento $b$ y la objetivo $\pi$ coincidían (pesos $W \approx 1$), reduciendo la varianza respecto al IS ordinario.

### **4.2 Propuestas para Estudios Futuros**

1. **Efecto de $\varepsilon$ en off-policy para Taxi**: con $\varepsilon$ bajo la política b es casi greedy, los pesos W son cercanos a 1 y la varianza cae, pero la exploración de PICKUP/DROPOFF es escasa. ¿Existe un $\varepsilon$ óptimo que equilibre exploración e IS?

2. **IS ordinario vs IS ponderado en Taxi**: implementar la variante con IS ordinario (divide por número de visitas) y comparar varianza y sesgo de las estimaciones Q.

3. **Entorno estocástico (`is_raining=True`)**: Taxi-v3 permite introducir incertidumbre en los movimientos (80% éxito). ¿Cómo afecta la estocasticidad a la brecha entre on-policy y off-policy?

4. **$\gamma < 1$ en Taxi**: con $\gamma = 1$ el agente valora igual recompensas lejanas y cercanas. Con $\gamma = 0.95$ se penalizan los caminos largos adicionalmente al −1/paso. ¿Converge antes con $\gamma < 1$?

5. **Comparativa con SARSA y Q-Learning**: los métodos TD actualizan Q en cada paso (no al final del episodio). ¿Cuánto más rápido convergen en Taxi, donde los episodios iniciales de 200 pasos retrasan enormemente el aprendizaje MC?